In [22]:
import torch
import torch.nn as nn
import torch.optim as optim

In [33]:
# --------------------------------------------------
# 1. Training data
# --------------------------------------------------
# 10 students × 8 features
#
# [attendance, study_hours, assignment,
#  midterm, previous_exam, participation,
#  sleep, practice_tests]

X = torch.tensor([
    [90, 5, 85, 80, 82, 90, 7, 8],
    [75, 2, 60, 55, 58, 65, 6, 4],
    [95, 6, 92, 88, 90, 95, 8, 10],
    [60, 1, 50, 45, 40, 50, 5, 2],
    [85, 4, 78, 75, 76, 80, 7, 7],
    [70, 2, 65, 50, 55, 60, 6, 3],
    [92, 5, 88, 90, 85, 92, 8, 9],
    [55, 1, 45, 40, 42, 45, 5, 1],
    [80, 3, 72, 70, 68, 75, 7, 5],
    [65, 2, 55, 48, 50, 55, 6, 2]
], dtype=torch.float32)
print(X.shape)

# 0 = Fail, 1 = Pass
y = torch.tensor([
    [1],
    [0],
    [1],
    [0],
    [1],
    [0],
    [1],
    [0],
    [1],
    [0]
], dtype=torch.float32)
print(y.shape)

torch.Size([10, 8])
torch.Size([10, 1])


In [34]:
# --------------------------------------------------
# 2. Normalize input
# --------------------------------------------------

X_norm = X / torch.tensor(
    [100, 10, 100, 100, 100, 100, 10, 10],
    dtype=torch.float32
)

In [27]:
X_norm

tensor([[0.9000, 0.5000, 0.8500, 0.8000, 0.8200, 0.9000, 0.7000, 0.8000],
        [0.7500, 0.2000, 0.6000, 0.5500, 0.5800, 0.6500, 0.6000, 0.4000],
        [0.9500, 0.6000, 0.9200, 0.8800, 0.9000, 0.9500, 0.8000, 1.0000],
        [0.6000, 0.1000, 0.5000, 0.4500, 0.4000, 0.5000, 0.5000, 0.2000],
        [0.8500, 0.4000, 0.7800, 0.7500, 0.7600, 0.8000, 0.7000, 0.7000],
        [0.7000, 0.2000, 0.6500, 0.5000, 0.5500, 0.6000, 0.6000, 0.3000],
        [0.9200, 0.5000, 0.8800, 0.9000, 0.8500, 0.9200, 0.8000, 0.9000],
        [0.5500, 0.1000, 0.4500, 0.4000, 0.4200, 0.4500, 0.5000, 0.1000],
        [0.8000, 0.3000, 0.7200, 0.7000, 0.6800, 0.7500, 0.7000, 0.5000],
        [0.6500, 0.2000, 0.5500, 0.4800, 0.5000, 0.5500, 0.6000, 0.2000]])

In [28]:
class StudentModel(nn.Module):
    def __init__(self):
        super().__init__()
        
        self.network = nn.Sequential(
            nn.Linear(8, 16),
            nn.ReLU(),
            nn.Linear(16, 8),
            nn.ReLU(),
            nn.Linear(8, 1),
        )
    def forward(self, x):
        return self.network(x)
     

In [38]:
# --------------------------------------------------
# 4. Training
# --------------------------------------------------

torch.manual_seed(42)

model = StudentModel()


for name, param in model.named_parameters():
    print(name)
    print(param)
    print("Shape:", param.shape)
    print("Gradient :", param.grad)
    print()


total_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print("Total trainable parameters:", total_params)



# BCEWithLogitsLoss includes Sigmoid internally.
criterion = nn.BCEWithLogitsLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.01
)

EPOCHS = 1000

for epoch in range(EPOCHS):
    logits = model(X_norm)

    # Calculate loss
    loss = criterion(logits, y)

    # Clear old gradients
    optimizer.zero_grad()

    # Backpropagation
    loss.backward()

    # Adam updates weights
    optimizer.step()

    if epoch % 50 == 0:
        print(f"Epoch {epoch:4d} | Loss: {loss.item():.4f}")

print(f"Epoch {EPOCHS:4d} | Loss: {loss.item():.4f}")

network.0.weight
Parameter containing:
tensor([[ 0.2703,  0.2935, -0.0828,  0.3248, -0.0775,  0.0713, -0.1721,  0.2076],
        [ 0.3117, -0.2594,  0.3073,  0.0662,  0.2612,  0.0479,  0.1705, -0.0499],
        [ 0.2725,  0.0523, -0.1651,  0.0901, -0.1629, -0.0415, -0.1436,  0.2345],
        [-0.2791, -0.1630, -0.0998, -0.2126,  0.0334, -0.3492,  0.3193, -0.3003],
        [ 0.2730,  0.0588, -0.1148,  0.2185,  0.0551,  0.2857,  0.0387, -0.1115],
        [ 0.0950, -0.0959,  0.1488,  0.3157,  0.2044, -0.1546,  0.2041,  0.0633],
        [ 0.1795, -0.2155, -0.3500, -0.1366, -0.2712,  0.2901,  0.1018,  0.1464],
        [ 0.1118, -0.0062,  0.2767, -0.2512,  0.0223, -0.2413,  0.1090, -0.1218],
        [ 0.1083, -0.0737,  0.2932, -0.2096, -0.2109, -0.2109,  0.3180,  0.1178],
        [ 0.3402, -0.2918, -0.3507, -0.2766, -0.2378,  0.1432,  0.1266,  0.2938],
        [-0.1826, -0.2410,  0.1876, -0.1429,  0.2146, -0.0839,  0.2022, -0.2747],
        [-0.1784,  0.1078,  0.0747, -0.0901,  0.2107,  0.24

In [39]:
# --------------------------------------------------
# 5. Prediction
# --------------------------------------------------
with torch.no_grad():

    logits = model(X_norm)   # must match what the model was trained on

    # Convert logits → probability
    probabilities = torch.sigmoid(logits)

    # Probability >= 0.5 → Pass
    predictions = (probabilities >= 0.5).float()

In [40]:
# --------------------------------------------------
# 6. Display results
# --------------------------------------------------

print("\nResults:")

for i in range(10):
    result = "PASS" if predictions[i] == 1 else "FAIL"
    print(
        f"Student {i+1}: "
        f"Probability = {probabilities[i].item():.3f}, "
        f"Prediction = {result}, "
        f"Actual = {'PASS' if y[i] == 1 else 'FAIL'}"
    )


Results:
Student 1: Probability = 0.986, Prediction = PASS, Actual = PASS
Student 2: Probability = 0.000, Prediction = FAIL, Actual = FAIL
Student 3: Probability = 0.986, Prediction = PASS, Actual = PASS
Student 4: Probability = 0.000, Prediction = FAIL, Actual = FAIL
Student 5: Probability = 0.986, Prediction = PASS, Actual = PASS
Student 6: Probability = 0.000, Prediction = FAIL, Actual = FAIL
Student 7: Probability = 0.986, Prediction = PASS, Actual = PASS
Student 8: Probability = 0.000, Prediction = FAIL, Actual = FAIL
Student 9: Probability = 0.986, Prediction = PASS, Actual = PASS
Student 10: Probability = 0.000, Prediction = FAIL, Actual = FAIL
